# Calidad del Vino — Predicción

Proyecto de ciencia de datos listo para **Google Colab**.

**Pregunta:** ¿podemos predecir la calidad de un vino a partir de sus propiedades fisicoquímicas?

**Target:** `quality` (puntuación de 0 a 10).

**Stack:** pandas, seaborn, scikit-learn, shap

Dataset clásico de UCI Wine Quality (rojo).

## 0. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
print("OK")

## 1. Cargar datos

In [ ]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
df = pd.read_csv(url, sep=";")
print(df.shape)
df.head()

## 2. EDA

In [ ]:
print(df["quality"].value_counts().sort_index())
df.describe().T

In [ ]:
plt.figure(figsize=(8, 4))
sns.countplot(data=df, x="quality")
plt.title("Distribución de calidad")
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Correlación")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=df, x="quality", y="alcohol", ax=axes[0])
axes[0].set_title("Alcohol vs Calidad")
sns.boxplot(data=df, x="quality", y="volatile acidity", ax=axes[1])
axes[1].set_title("Acidez volátil vs Calidad")
plt.tight_layout()
plt.show()

### Hallazgos
- La mayoría de los vinos tienen calidad 5 o 6.
- `alcohol` correlaciona positivamente con calidad.
- `volatile acidity` correlaciona negativamente.

## 3. Preparar datos

In [ ]:
X = df.drop(columns="quality")
y = df["quality"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print(X_train.shape, X_test.shape)

## 4. Modelos (regresión)

In [ ]:
modelos = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(random_state=RANDOM_STATE),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE),
    "Gradient Boosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
}

resultados = []
pipes = {}

for nombre, clf in modelos.items():
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", clf)
    ])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    resultados.append({
        "modelo": nombre,
        "MAE": mean_absolute_error(y_test, y_pred),
        "RMSE": mean_squared_error(y_test, y_pred, squared=False),
        "R2": r2_score(y_test, y_pred),
    })
    pipes[nombre] = pipe

pd.DataFrame(resultados).sort_values("R2", ascending=False)

## 5. Feature importance

In [ ]:
rf = pipes["Random Forest"].named_steps["model"]
importancias = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(x=importancias.values, y=importancias.index)
plt.title("Importancia — Random Forest")
plt.tight_layout()
plt.show()

## 6. SHAP

In [ ]:
!pip install -q shap

In [ ]:
import shap

rf_pipe = pipes["Random Forest"]
X_train_scaled = rf_pipe.named_steps["scaler"].transform(X_train)
model = rf_pipe.named_steps["model"]

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_train_scaled)

print("SHAP listo")

In [ ]:
shap.summary_plot(shap_values, X_train_scaled, feature_names=X.columns, show=False)
plt.title("SHAP Summary — Calidad del vino")
plt.tight_layout()
plt.show()

In [ ]:
shap.summary_plot(shap_values, X_train_scaled, feature_names=X.columns, plot_type="bar", show=False)
plt.title("Importancia media SHAP")
plt.tight_layout()
plt.show()

## 7. Conclusiones

- El alcohol es la variable más importante para predecir calidad.
- Random Forest / Gradient Boosting suelen dar el mejor R² (~0.4–0.5 en test).
- El problema es de regresión con target ordinal; se puede binarizar (calidad alta ≥7) para clasificación.
- Ideal como proyecto de regresión después de Iris/Titanic.

**Rango esperado:** R² test ~0.40–0.50 | MAE ~0.5